In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import QuantileRegressor
from sklearn.metrics import mean_pinball_loss

In [3]:
df_master = pd.read_csv("historical_master_data.csv")

In [4]:
print(df_master.index.min(), df_master.index.max())
print(df_master.notna().sum())
print(df_master.tail(3))

0 59606
dt_utc           59607
T2M              13132
WS10             13079
load             59291
residual_load    59291
dtype: int64
                          dt_utc  T2M  WS10  load  residual_load
59604  2026-10-19 21:00:00+00:00  9.9   3.4   NaN            NaN
59605  2026-10-19 22:00:00+00:00  9.0   3.9   NaN            NaN
59606  2026-10-19 23:00:00+00:00  8.0   3.5   NaN            NaN


In [5]:
df = df_master.dropna().copy()
print(len(df), df.index.min(), df.index.max())

12760 46407 59292


In [9]:
def add_features(df):
    d: pd.DataFrame = df.copy()
    d = d.set_index(pd.DatetimeIndex(d), drop=False, inplace=True)
    idx = d.index.tz_convert('Europe/Berlin')  # calendar features in local time
    d['hour']    = idx.hour
    d['weekday'] = idx.weekday          # 0=Mon ... 6=Sun
    d['doy']     = idx.dayofyear
    d['is_fri']  = (d['weekday'] == 4).astype(int)
    d['is_sat']  = (d['weekday'] == 5).astype(int)
    # lags: 1h, 24h, 168h (1 week) — enough for a linear baseline
    for lag in [1, 24, 168]:
        d[f'lag{lag}'] = d['load'].shift(lag)
    return d.dropna()

df_feat = add_features(df)
print(df_feat.shape)
print(df_feat.head(2))

ValueError: iterator too short: Expected 12760 but iterator had only 5 items.

In [ ]:
def target_hours(issue_date_local):
    """Return the 6 target timestamps (UTC) for a submission issued on issue_date_local (a date, Wed)."""
    targets_local = []
    for day_offset in [2, 3]:           # Friday, Saturday
        for hh in [12, 16, 20]:
            targets_local.append(pd.Timestamp(issue_date_local) + pd.Timedelta(days=day_offset, hours=hh))
    ts_local = pd.DatetimeIndex(targets_local).tz_localize('Europe/Berlin')
    return ts_local.tz_convert('UTC')

# Pick a test issue date inside the overlap window
overlap_days = pd.date_range(df_feat.index.tz_convert('Europe/Berlin').normalize().min(),
                             df_feat.index.tz_convert('Europe/Berlin').normalize().max(), freq='W-WED')
issue = overlap_days[2]   # third Wednesday in the overlap
targets_utc = target_hours(issue)
print("issue (local):", issue)
print("targets (UTC):", targets_utc)

In [ ]:
QUANTILES = [0.025, 0.25, 0.5, 0.75, 0.975]
FEATURES  = ['hour', 'weekday', 'is_fri', 'is_sat', 'lag1', 'lag24', 'lag168']

# Training cutoff = Wednesday 23:59 local, converted to UTC
cutoff_utc = pd.Timestamp(issue, tz='Europe/Berlin') + pd.Timedelta(hours=23, minutes=59)
cutoff_utc = cutoff_utc.tz_convert('UTC')

train = df_feat[df_feat.index < cutoff_utc]
print("train rows:", len(train))

models = {}
for q in QUANTILES:
    m = QuantileRegressor(quantile=q, alpha=0.0, solver='highs')
    m.fit(train[FEATURES], train['load'])
    models[q] = m

In [ ]:
last_known_load = train['load'].iloc[-1]

X_pred = pd.DataFrame({
    'hour':    targets_utc.tz_convert('Europe/Berlin').hour,
    'weekday': targets_utc.tz_convert('Europe/Berlin').weekday,
    'is_fri':  (targets_utc.tz_convert('Europe/Berlin').weekday == 4).astype(int),
    'is_sat':  (targets_utc.tz_convert('Europe/Berlin').weekday == 5).astype(int),
    'lag1':    last_known_load,
    'lag24':   last_known_load,
    'lag168':  last_known_load,
}, index=targets_utc)

forecast = pd.DataFrame({f'q{q}': models[q].predict(X_pred[FEATURES]) for q in QUANTILES}, index=targets_utc)
forecast['load_actual'] = df_master['load'].reindex(targets_utc)
print(forecast)

In [ ]:
def quantile_score(y, q_hat, q):
    return np.where(y >= q_hat, q * (y - q_hat), (1 - q) * (q_hat - y))

mask = forecast['load_actual'].notna()
if mask.any():
    scores = []
    for q in QUANTILES:
        s = quantile_score(forecast.loc[mask, 'load_actual'].values,
                           forecast.loc[mask, f'q{q}'].values, q)
        scores.append(s.mean())
    print("mean quantile score per level:", dict(zip(QUANTILES, np.round(scores, 3))))
    print("avg linQS (approx CRPS):", np.mean(scores).round(3))
else:
    print("No actuals for these targets — pick an earlier issue date.")